In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('creditcard.csv')
df = df.drop_duplicates()
df.shape

In [ ]:
df['Class'].value_counts()

In [ ]:
df['Class'].value_counts().plot(kind='bar')
plt.title('Class Distribution (0 equals to Legit, 1 equals to Fraud)')
plt.xlabel('Class')
plt.ylabel('Count')
plt.show()

In [ ]:
df['Class'].value_counts().plot(kind='bar')
plt.title('Class Distribution Of Log Scale')
plt.xlabel('Class')
plt.ylabel('Count (log scale)')
plt.yscale('log')
plt.show()

In [ ]:
sns.histplot(data=df, x='Amount', hue='Class', stat='density', common_norm=False, bins=50, binrange=(0, 500))
plt.title('Transaction Amount Distribution: Fraud vs Legit')
plt.xlabel('Amount')
plt.show()

In [ ]:
df['Hour'] = (df['Time'] % 86400) / 3600
sns.histplot(data=df, x='Hour', hue='Class', stat='density', common_norm=False, bins=48)
plt.title('Transaction Hour of Day: Fraud vs Legit')
plt.xlabel('Hour of Day')
plt.show()

In [ ]:
correlations = df.corr()['Class'].sort_values()
print(correlations)

In [ ]:
top_features = ['V17', 'V14', 'V12', 'V10']

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()

for i, feature in enumerate(top_features):
    sns.histplot(data=df, x=feature, hue='Class', stat='density', common_norm=False, ax=axes[i])
    axes[i].set_title(f'{feature} Distribution: Fraud vs Legit')

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop('Class', axis=1)
y = df['Class']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(y_train.value_counts())
print(y_test.value_counts())

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

model_baseline = LogisticRegression(max_iter=1000)
model_baseline.fit(X_train, y_train)

y_pred_baseline = model_baseline.predict(X_test)

print(confusion_matrix(y_test, y_pred_baseline))
print(classification_report(y_test, y_pred_baseline))

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
model_baseline_scaled = LogisticRegression(max_iter=1000)
model_baseline_scaled.fit(X_train_scaled, y_train)

y_pred_scaled = model_baseline_scaled.predict(X_test_scaled)

print(confusion_matrix(y_test, y_pred_scaled))
print(classification_report(y_test, y_pred_scaled))

In [ ]:
model_weighted = LogisticRegression(max_iter=1000, class_weight='balanced')
model_weighted.fit(X_train_scaled, y_train)

y_pred_weighted = model_weighted.predict(X_test_scaled)

print(confusion_matrix(y_test, y_pred_weighted))
print(classification_report(y_test, y_pred_weighted))

In [ ]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train_scaled, y_train)

print(y_train_smote.value_counts())

In [ ]:
model_smote = LogisticRegression(max_iter=1000)
model_smote.fit(X_train_smote, y_train_smote)

y_pred_smote = model_smote.predict(X_test_scaled)

print(confusion_matrix(y_test, y_pred_smote))
print(classification_report(y_test, y_pred_smote))

In [ ]:
from sklearn.ensemble import RandomForestClassifier

model_rf = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42, n_jobs=-1)
model_rf.fit(X_train_scaled, y_train)

y_pred_rf = model_rf.predict(X_test_scaled)

print(confusion_matrix(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))

In [ ]:
from sklearn.ensemble import RandomForestClassifier

model_rf = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42, n_jobs=-1)
model_rf.fit(X_train_scaled, y_train)

y_pred_rf = model_rf.predict(X_test_scaled)

print(confusion_matrix(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))

In [ ]:
importances = pd.Series(model_rf.feature_importances_, index=X_train.columns).sort_values(ascending=False)
print(importances.head(10))

In [ ]:
from xgboost import XGBClassifier

scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

model_xgb = XGBClassifier(scale_pos_weight=scale_pos_weight, random_state=42, eval_metric='logloss')
model_xgb.fit(X_train_scaled, y_train)

y_pred_xgb = model_xgb.predict(X_test_scaled)

print(confusion_matrix(y_test, y_pred_xgb))
print(classification_report(y_test, y_pred_xgb))

In [ ]:
from sklearn.metrics import precision_recall_curve, PrecisionRecallDisplay

y_scores_rf = model_rf.predict_proba(X_test_scaled)[:, 1]
y_scores_xgb = model_xgb.predict_proba(X_test_scaled)[:, 1]

precision_rf, recall_rf, _ = precision_recall_curve(y_test, y_scores_rf)
precision_xgb, recall_xgb, _ = precision_recall_curve(y_test, y_scores_xgb)

plt.figure(figsize=(8, 6))
plt.plot(recall_rf, precision_rf, label='Random Forest')
plt.plot(recall_xgb, precision_xgb, label='XGBoost')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision and Recall: Random Forest vs XGBoost')
plt.legend()
plt.show()